In [ ]:
import time
NOTEBOOK_START_TIME = time.time()
NOTEBOOK_MAX_ALLOWED_TIME = 60 * 60 * 22 # 22 hours
MAX_ALLOWED_TIME_PER_SAMPLE = 60 * 10 # 10 minutes

In [ ]:
!cp -r /kaggle/usr/lib/install_llama_cpp_l4/ .
!chmod +x install_llama_cpp_l4/llama.cpp/build/bin/llama-server

In [ ]:
import subprocess
import time
import os
import signal

# Function to start the server
def start_llama_server(gpu, port, ctx, seed, is_reasoner):
    # Start the server process
    if is_reasoner:
        cmd = f'CUDA_VISIBLE_DEVICES={gpu} LD_LIBRARY_PATH=$LD_LIBRARY_PATH:./install_llama_cpp_l4/llama.cpp/build/bin/ ./install_llama_cpp_l4/llama.cpp/build/bin/llama-server --model /kaggle/input/qwq-unsloth/QwQ-32B-Q4_K_M.gguf --port {port} --threads 12 --ctx-size {ctx} --n-gpu-layers 99 --seed {seed} --no-mmap --prio 2 --temp 0.6 --repeat-penalty 1.0 --dry-multiplier 0.5 --min-p 0.0 --top-k 40 --top-p 0.95 --samplers "top_k;top_p;min_p;temperature;dry;typ_p;xtc" -fa'
    else:
        cmd = f"CUDA_VISIBLE_DEVICES={gpu} LD_LIBRARY_PATH=$LD_LIBRARY_PATH:./install_llama_cpp_l4/llama.cpp/build/bin/ ./install_llama_cpp_l4/llama.cpp/build/bin/llama-server --model /kaggle/input/barto-qwen-coder/Barto-Qwen2.5-Coder-32B-Instruct-Q4_K_M.gguf --port {port} --threads 12 --ctx-size {ctx} --n-gpu-layers 99 --seed {seed} --no-mmap --prio 2 --temp 0.8 --min-p 0.01 --top-k 40 --top-p 0.9 -fa"
    # Start process and don't wait for it to complete
    print(cmd)
    process = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        preexec_fn=os.setsid  # This allows killing the process group later
    )
    
    print(f"Started llama-server with PID: {process.pid}")
    return process

In [ ]:
# Start the server
server_process0 = start_llama_server(gpu='0,1', port=8091, ctx=32_000, seed=123, is_reasoner=False)
server_process1 = start_llama_server(gpu='2', port=8092, ctx=14_000, seed=666, is_reasoner=True)
server_process2 = start_llama_server(gpu='3', port=8093, ctx=14_000, seed=777, is_reasoner=True)
# server_process1 = start_llama_server(1, 8092)

# To stop the server later, call:
# stop_llama_server(server_process)

In [ ]:
# Function to stop the server when needed
def stop_llama_server(process):
    try:
        if process:
            os.killpg(os.getpgid(process.pid), signal.SIGTERM)
            print("Server stopped")
    except:
        print("Failed to stop server")

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "hf_..."

In [ ]:
!cp -r /kaggle/input/custom-agentless-cot/* /kaggle/working

In [ ]:
!echo $PYTHONPATH

In [ ]:
!pip install -q -t /kaggle/working/site-packages /kaggle/input/eduardo-wheels/wheels/*.whl
# !pip download -r requirements.txt -d ./wheels
# !zip -r wheels.py wheels
# !pip install -t /kaggle/working/site-packages -r requirements.txt

In [ ]:
import subprocess
from pathlib import Path


class PythonManager:
    @staticmethod
    def install_cmds() -> list[str]:
        return ["apt install -y python3.11 python3.11-dev python3.11-venv"]

    @staticmethod
    def get_ubuntu_version() -> str:
        result = subprocess.run(
            "lsb_release -rs",
            shell=True,
            executable="/bin/bash",
            capture_output=True,
            text=True,
        )
        return result.stdout.strip()

    @staticmethod
    def install_offline_cmds(
        python_debs_dir: Path,
        ubuntu_version: str = "22.04",
    ) -> list[str]:
        python_debs_dir = Path(python_debs_dir)
        return [
            f"dpkg -i {python_debs_dir}/ubuntu_{ubuntu_version}/*.deb"
        ]

def install_python311():
    cmd = PythonManager.install_offline_cmds("/kaggle/input/konwinski-prize/kprize_setup/python3.11", PythonManager.get_ubuntu_version())[0]
    result = subprocess.run(
                cmd,
                shell=True,
                executable="/bin/bash",
                capture_output=True,
                text=True,
            )
    print(result.stdout)
    print(result.stderr)

install_python311()

In [ ]:
# !pip install -q --no-index --target=/kaggle/working /kaggle/input/vllm-wheels/vllm_wheels/*.whl

In [ ]:
import sys
sys.path.append('/kaggle/working/site-packages')

In [ ]:
!mkdir -p /kaggle/working/site-packages/llama_index/core/_static/nltk_cache/tokenizers/punkt/PY3_tab

In [ ]:
import os
# https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/discussion/560682#3113134
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["MY_LOG_LEVEL"] = "40" # error

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
print(len(os.sched_getaffinity(0)))

In [ ]:
# !pip download openai==1.61.1 -d openai_whl

In [ ]:
!pip install --no-index  /kaggle/input/openai-whl/openai_whl/*.whl

In [ ]:
import shutil
import time
import pandas as pd
import polars as pl
import io

import argparse
from pathlib import Path
from queue import Queue
import threading
import time
import json
from git import Repo
from pathlib import Path
import subprocess
import xml.etree.ElementTree as ET

import main
from agentless.fl.localize import localize
from agentless.test.select_pass_to_pass_tests import get_test_files
from agentless.util.preprocess_data import get_repo_structure
from agentless.util.utils import setup_logger

import kaggle_evaluation.konwinski_prize_inference_server

The evaluation API requires that you set up a server which will respond to inference requests. We have already defined the server; you just need write the predict function. When we evaluate your submission on the hidden test set the client defined in `konwinski_prize_gateway` will run in a different container with direct access to the hidden test set and hand off the data.

Your code will always have access to the published copies of the files.

In [ ]:
instance_count = None

def get_number_of_instances(num_instances: int) -> None:
    """ The very first message from the gateway will be the total number of instances to be served.
    You don't need to edit this function.
    """
    global instance_count
    instance_count = num_instances

In [ ]:
REPO_PATH = 'repo'
REPO_CLONE_PATH = 'repo_clone'
RESULT_PATH = 'result'
PARQUET_PATH = 'data.parquet'

In [ ]:
def clean_dirs():
    paths_to_clean = [
        "./pip_packages",
        REPO_PATH,
        REPO_CLONE_PATH,
        RESULT_PATH,
        PARQUET_PATH,
        "./playground",
        "./embedding",
        "./git.patch",
    ]
    
    for path_str in paths_to_clean:
        path = Path(path_str)
        try:
            if path.exists():
                if path.is_file():
                    path.unlink()
                else:
                    shutil.rmtree(path, ignore_errors=True)
        except Exception as e:
            print(f"Error while removing {path}: {e}")
            continue

# Init LLM

In [ ]:
TEMPERATURE = 0.8
MIN_P = 0.01
TOP_P = 0.9
TOP_K = 40

USE_MEMORY = False
FINE_GRAIN_NB_SAMPLES=2
REPAIR_NB_SAMPLES=3
TEST_NB_SAMPLES=3
# TEST_SAMPLE_BATCH_SIZE=5

INSTALL_TIMEOUT = 60 * 3
TESTS_TIMEOUT = 60 * 2
UNIT_TEST_TIMEOUT = 60 * 2

SEED = 6335

In [ ]:
from agentless.util.model import LlamaCppLLM
import warnings
import os
from transformers import AutoTokenizer

warnings.simplefilter('ignore')

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def get_tokenizer(model_path):
    # kwargs = {}
    # kwargs["gguf_file"] = Path(model_path).name
    # model_path = Path(model_path).parent

    tokenizer = AutoTokenizer.from_pretrained(
                    "/kaggle/input/qwen2.5-coder/transformers/32b-instruct-awq/1",
                    # **kwargs
                )    
    return tokenizer

llm_model_pth = '/kaggle/input/barto-qwen-coder/Qwen2.5-Coder-32B-Instruct-Q8_0.gguf'
llm = LlamaCppLLM(llm_model_pth, get_tokenizer(llm_model_pth), port=8091, is_reasoner=False, temperature=TEMPERATURE, top_p=TOP_P, min_p=MIN_P, top_k=TOP_K)
llm_qwq1 = LlamaCppLLM(llm_model_pth, get_tokenizer(llm_model_pth), port=8092, is_reasoner=True, temperature=TEMPERATURE, top_p=TOP_P, min_p=MIN_P, top_k=TOP_K)
llm_qwq2 = LlamaCppLLM(llm_model_pth, get_tokenizer(llm_model_pth), port=8093, is_reasoner=True, temperature=TEMPERATURE, top_p=TOP_P, min_p=MIN_P, top_k=TOP_K)

In [ ]:
def is_model_ready(llm):
    # make a curl to the port and check if the model is ready
    import requests

    url = f"http://localhost:{llm.port}/health"
    try:
        response = requests.get(url).json()
        if response["status"] == "ok":
            return True
        else:
            return False
    except Exception as e:
        return False

In [ ]:
embd_model = None

# Helper Functions

In [ ]:
def init_working_repo():
    if os.path.exists(REPO_CLONE_PATH):
        shutil.rmtree(REPO_CLONE_PATH, ignore_errors=True)
    shutil.copytree(REPO_PATH, REPO_CLONE_PATH)
    
    # Remove any existing git folder
    git_path = Path(REPO_CLONE_PATH) / ".git"
    if git_path.exists():
        shutil.rmtree(git_path, ignore_errors=True)
    
    # Init new repo and commit everything
    repo = Repo.init(REPO_CLONE_PATH)
    repo.index.add('*')  # Add all files
    repo.index.commit("Initial commit")
    return repo

def load_jsonl(sample_path):
    # Load the JSONL file
    samples = []
    with open(sample_path, 'r') as f:
        for line in f:
            samples.append(json.loads(line))
    return samples

def load_repair_samples(
        result_folder, num_samples=FINE_GRAIN_NB_SAMPLES, 
        num_outputs_per_sample=REPAIR_NB_SAMPLES,
        base_path="repair_sample_{sample_idx}/output_{output_idx}_processed.jsonl"):
    samples = []
    for sample_idx in range(1, num_samples + 1):
        for output_idx in range(num_outputs_per_sample):
            file_path = base_path.format(sample_idx=sample_idx, output_idx=output_idx)
            if os.path.exists(os.path.join(result_folder, file_path)):
                samples.extend(load_jsonl(os.path.join(result_folder, file_path)))
    return samples

def fix_repo_paths_in_patches(patch):
    return patch.replace("repo_clone/", "")

def _apply_patch(patch_path: Path, repo_path: Path, timeout: int=15, is_dry_run: bool=True) -> bool:
    patch_path = Path(patch_path)
    repo_path = Path(repo_path)
    dry_run_cmd = f"--dry-run" if is_dry_run else ""
    cmd = f"patch --quiet {dry_run_cmd} -p1 -i {patch_path.resolve()} -d {repo_path.resolve()}"
    try:
        result = subprocess.run(cmd, shell=True, check=False, text=True, capture_output=True, timeout=timeout)
        # print(f"stdout: {result.stdout}\nstderr: {result.stderr}")
        if result.returncode == 0:
            return True
        else:
            return False
    except subprocess.CalledProcessError:
        return False

def apply_patch(patch_str, repo_path, is_dry_run: bool=True, patch_path: str="./git.patch"):
    patch_path = Path(patch_path).resolve()
    if patch_path.exists():
        patch_path.unlink(missing_ok=True)

    patch_path.write_text(patch_str)
    result = _apply_patch(patch_path, repo_path, is_dry_run=is_dry_run)
    patch_path.unlink(missing_ok=True)
    return result

def get_valid_patches():
    patches = list(set([x["model_patch"] for x in load_repair_samples(RESULT_PATH)]))
    patches = [fix_repo_paths_in_patches(p) for p in patches if p]
    patches = [patch for patch in patches if apply_patch(patch, REPO_CLONE_PATH, is_dry_run=True)]
    return patches

# Regression tests functions

In [ ]:
def exec_run_with_timeout(cmd, timeout: int|None=60, logger=None):
    # Local variables to store the result of executing the command
    exec_result = ''
    exec_id = None
    exception = None
    timed_out = False
    process = None

    # Wrapper function to run the command
    def run_command():
        current_env = os.environ.copy()
        jupyter_vars = ['MPLBACKEND', 'JPY_PARENT_PID', 'JUPYTER_CONFIG_DIR', 'JUPYTER_PATH', 'JUPYTER_DATA_DIR']
        clean_env = {k: v for k, v in current_env.items() if k not in jupyter_vars}
                
        nonlocal exec_result, exec_id, exception, process
        try:
            process = subprocess.run(cmd, shell=True, executable="/bin/bash", check=False, text=True, capture_output=True, timeout=timeout, env=clean_env)
            exec_result = process.stdout
            logger.info(f"stdout: {process.stdout}\nstderr: {process.stderr}")
        except Exception as e:
            logger.info(f"Error executing command: {e}")
            try:
                logger.info(f"stdout: {process.stdout}\nstderr: {process.stderr}")
            except:
                pass
            exception = e

    # Start the command in a separate thread
    thread = threading.Thread(target=run_command)
    start_time = time.time()
    thread.start()
    thread.join(timeout)

    if exception:
        print(exception)
        return None, True, time.time() - start_time

    # If the thread is still alive, the command timed out
    if thread.is_alive():
        # what do I do here for subprocess.run ?
        # I need to kill the process
        # if process:
        #     process.kill()
        timed_out = True
    end_time = time.time()
    return exec_result, timed_out, end_time - start_time

def load_regression_samples(
        result_folder,
        num_test_samples=TEST_NB_SAMPLES,
        base_path="reproduction_test_samples/output_{output_idx}_processed_reproduction_test.jsonl"):
    samples = []
    for output_idx in range(num_test_samples):
        file_path = base_path.format(output_idx=output_idx)
        if os.path.exists(os.path.join(result_folder, file_path)):
            samples.extend(load_jsonl(os.path.join(result_folder, file_path)))
    return samples

def get_valid_regression_tests():
    patches = list(set([x["test_patch"] for x in load_regression_samples(RESULT_PATH, base_path="reproduction_test_samples/output_{output_idx}_processed_reproduction_test.jsonl")] + [x["test_patch"] for x in load_regression_samples(RESULT_PATH, base_path="reproduction_test_samples_qwq1/output_{output_idx}_processed_reproduction_test.jsonl")] + [x["test_patch"] for x in load_regression_samples(RESULT_PATH, base_path="reproduction_test_samples_qwq2/output_{output_idx}_processed_reproduction_test.jsonl")]))
    patches = [fix_repo_paths_in_patches(p) for p in patches if p]
    patches = [patch for patch in patches if apply_patch(patch, REPO_CLONE_PATH, is_dry_run=True)]
    return patches
    
def filter_regression_tests(regression_tests, repo_dir, expected_output, unexpected_outputs, logger):
    start_time = time.time()
    outs = []
    for i, test in enumerate(regression_tests):
        elapsed_time = time.time() - start_time
        timeout = max(0, TESTS_TIMEOUT - elapsed_time)
        if timeout <= 0:
            break
        
        bug_path = os.path.join(repo_dir, 'reproduce_bug.py')

        # Remove the reproduce_bug.py file
        if os.path.exists(bug_path):
            os.remove(bug_path)
            
        res = apply_patch(test, repo_dir, is_dry_run=False)
        if not res:
            continue
        cmd = f'cd {repo_dir} && . .venv/bin/activate && python reproduce_bug.py'
        logger.info(f"Testing regression test {i}:")
        result, timed_out, duration = exec_run_with_timeout(cmd, timeout=timeout, logger=logger)
        
        # Remove the reproduce_bug.py file
        if os.path.exists(bug_path):
            os.remove(bug_path)
            
        # print(i, result, timed_out, duration)
        failed = False
        if not timed_out:
            for unexpected_output in unexpected_outputs:
                if unexpected_output.lower() in result.lower():
                    failed = True
                    break
            if not failed and expected_output.lower() in result.lower():
                outs.append(test)
    return outs

def get_regression_tests_that_reproduced_issue(logger):
    tests = get_valid_regression_tests()
    if not tests:
        return []
    logger.info(f"Filtering regression tests that reproduced issue")
    tests = filter_regression_tests(tests, REPO_CLONE_PATH, expected_output="Issue reproduced", unexpected_outputs=["Issue resolved", "Other issues"], logger=logger)
    return tests

def select_patch_with_most_regression_tests_fixed(repo, patch_candidates, regression_tests, src_dir_with_installed_packages=REPO_CLONE_PATH, logger=None):
    fail_to_pass_patches = []

    for i, patch in enumerate(patch_candidates):
        try:
            # More thorough cleanup before testing each patch
            repo.head.reset(index=True, working_tree=True)
            repo.index.checkout(force=True)
            # repo.git.clean('-fd')  # Remove untracked files and directories

            # Apply repair patch
            res = apply_patch(patch, src_dir_with_installed_packages, is_dry_run=False)
            if not res:
                print(f"Failed to apply patch {i}")
                continue
            
            # Run regression tests
            logger.info(f"Testing F2P for patch {i}")
            tests = filter_regression_tests(
                regression_tests, 
                src_dir_with_installed_packages, 
                expected_output="Issue resolved", 
                unexpected_outputs=["Issue reproduced", "Other issues"],
                logger=logger
            )

            if len(tests) > 0:
                print(f"Patch {i} fixed the issue for {len(tests)} regression tests!!")
                fail_to_pass_patches.append((patch, len(tests)))

        except Exception as e:
            print(f"Error processing patch {i}: {str(e)}")
            continue
            
        finally:
            # Always ensure cleanup, even if an error occurred
            try:
                repo.head.reset(index=True, working_tree=True)
                repo.index.checkout(force=True)
                # repo.git.clean('-fd')
            except Exception as e:
                print(f"Error during cleanup after patch {i}: {str(e)}")

    # Select the patches sorted by number of fixed tests
    if fail_to_pass_patches:
        fail_to_pass_patches = sorted(fail_to_pass_patches, key=lambda x: x[1], reverse=True)
        return [x[0] for x in fail_to_pass_patches]
    return None

# Installer Manager

In [ ]:
class Installer:
    def __init__(self, pip_packages_archive, env_setup_cmds_templates, timeout,
               repo_path=REPO_CLONE_PATH, tmp_pip_packages_path='pip_packages',
               pip_archive_dir= '/tmp/pip_packages_archive.tar', logger=None):
        self.pip_packages_archive = pip_packages_archive
        self.env_setup_cmds_templates = env_setup_cmds_templates
        self.timeout = timeout
        self.repo_path = repo_path
        self.tmp_pip_packages_path = tmp_pip_packages_path
        self.pip_archive_dir = pip_archive_dir
        self.thread = threading.Thread(target=self.run)
        self.start_time = None
        self.logger = logger

    def start(self):
        self.start_time = time.time()
        self.thread.start()

    def join(self):
        elapsed_time = time.time() - self.start_time
        self.thread.join(max(0, self.timeout - elapsed_time))
        if self.thread.is_alive():
            return False
        return True
    
    @staticmethod
    def sanitize_env_commands(env_setup_cmds):
        def add_options(cmd, opt):
            if opt not in cmd:
                return f" {opt} "
            return ""

        out = []
        for cmd in env_setup_cmds:
            out_inner = []
            for c in cmd.split("&&"):
                new_c = c
                if "uv venv" in c:
                    to_replace = "uv venv"
                    to_replace = to_replace + add_options(c, "--no-python-downloads")
                    to_replace = to_replace + add_options(c, "--no-index")
                    new_c = new_c.replace("uv venv", to_replace)
                elif "uv pip install" in c:
                    to_replace = "uv pip install"
                    to_replace = to_replace + add_options(c, "--no-index")
                    to_replace = to_replace + add_options(c, "--find-links={pip_packages_path}")
                    new_c = new_c.replace("uv pip install", to_replace)
                out_inner.append(new_c)
            out.append(" && ".join(out_inner))

        uv_install_cmd = "uv pip install uv --no-index --find-links=/kaggle/input/uv-whl"
        uv_install_dev = "uv pip install pylint pytest black flake8 isort --no-index --find-links=/kaggle/input/dev-wheels/wheels"
        out = out[:-1] + [uv_install_cmd, uv_install_dev] + out[-1:]
        return out    
    
    def run(self):
        repo_path = str(Path(self.repo_path).resolve())
        tmp_pip_packages_path = str(Path(self.tmp_pip_packages_path).resolve())

        # Extract the pip packages
        if self.pip_packages_archive is not None:
            if os.path.exists(self.pip_archive_dir):
                os.remove(self.pip_archive_dir)
                
            with open(self.pip_archive_dir, 'wb') as f:
                f.write(self.pip_packages_archive.read())
        
            if os.path.exists(tmp_pip_packages_path):
                shutil.rmtree(tmp_pip_packages_path, ignore_errors=True)
            shutil.unpack_archive(self.pip_archive_dir, extract_dir=tmp_pip_packages_path)
            os.remove(self.pip_archive_dir)

        # Get env setup cmds by setting the pip_packages_path
        env_setup_cmds_templates = Installer.sanitize_env_commands(self.env_setup_cmds_templates)
        prefix_cmd = [] # ["export PYTHONNOUSERSITE=1", 'export PYTHONPATH=""'] # avoid conflict by looking to other PYTHONPATHs
        env_setup_cmds = [cmd.format(pip_packages_path=tmp_pip_packages_path) for cmd in prefix_cmd + env_setup_cmds_templates]
        print(env_setup_cmds)
        # Run env setup for the repo
        try:
            start_time = time.time()
            result = subprocess.run(
                "\n".join(env_setup_cmds),
                shell=True,
                executable="/bin/bash",
                cwd=repo_path,
                capture_output=True,
                text=True,
                timeout=self.timeout,
                check=True
            )
            print(f"Finshed installing in {(time.time() - start_time) / 60}min")
            self.logger.info(f"Finshed installing in {(time.time() - start_time) / 60}min")
            self.logger.info(result.stdout)
            self.logger.info(result.stderr)

        except Exception as e:
            try:
                self.logger.info(f"Error installing packages: {e}")
                self.logger.info(result.stdout)
                self.logger.info(result.stderr)            
            except:
                pass
            print(e)

# Unit tests

In [ ]:
class UnitTestRunner:
    def __init__(self, repo_path, file_paths, patch=None, timeout=60 * 2, verbose=2, logger=None):
        self.repo_path = Path(repo_path).resolve()
        self.file_paths = file_paths
        self.timeout = timeout
        self.verbose = verbose
        self.patch = patch
        self.queue = Queue()
        self.thread = threading.Thread(target=self.run)
        self.start_time = None
        self.logger = logger
    def start(self):
        self.thread.start()
        self.start_time = time.time()

    def join(self):
        elapsed_time = time.time() - self.start_time
        self.thread.join(max(0, self.timeout - elapsed_time))
        if self.thread.is_alive():
            return None
        return self.queue.get()

    def run(self):
        # Apply the patch
        if self.patch:
            apply_patch(self.patch, str(self.repo_path), is_dry_run=False)

        all_total, all_failures, all_errors, all_skipped, all_passed = 0, 0, 0, 0, 0
        for filepath in self.file_paths:
            filepath = Path(filepath).resolve()
            self.logger.info(f"Running unit tests for {filepath}")
            print(f"Running unit tests for {filepath}")

            # Create a temporary file for the junit XML output
            junit_path = (self.repo_path / 'junit.xml').resolve()
            if junit_path.exists():
                junit_path.unlink()
    
            current_env = os.environ.copy()
            jupyter_vars = ['MPLBACKEND', 'JPY_PARENT_PID', 'JUPYTER_CONFIG_DIR', 'JUPYTER_PATH', 'JUPYTER_DATA_DIR']
            clean_env = {k: v for k, v in current_env.items() if k not in jupyter_vars}
            
            cmd = f". .venv/bin/activate && timeout -k 5 {self.timeout}s pytest -rA --tb=no --junitxml={str(junit_path)} {' '.join(self.file_paths)}"
    
            result = subprocess.run(
                cmd,
                shell=True,
                executable="/bin/bash",
                cwd=str(self.repo_path),
                capture_output=True,
                text=True,
                timeout=self.timeout,
                env=clean_env
            )
            if self.verbose > 1:
                print(result.stdout)
                print(result.stderr)
    
            # Parse the junit XML to get test counts
            if junit_path.exists():
                tree = ET.parse(junit_path)
                root = tree.getroot()
                testsuite = root.find('testsuite')
                if testsuite is not None:
                    total = int(testsuite.get('tests', 0))
                    failures = int(testsuite.get('failures', 0))
                    errors = int(testsuite.get('errors', 0))
                    skipped = int(testsuite.get('skipped', 0))
                    passed = total - failures - errors - skipped
                    
                    self.logger.info(f"Test Results:")
                    self.logger.info(f"Passed: {passed}")
                    self.logger.info(f"Failed: {failures}")
                    self.logger.info(f"Errors: {errors}")
                    self.logger.info(f"Skipped: {skipped}")
                    self.logger.info(f"Total: {total}")
                    if self.verbose:
                        print(f"Test Results:")
                        print(f"Passed: {passed}")
                        print(f"Failed: {failures}")
                        print(f"Errors: {errors}")
                        print(f"Skipped: {skipped}")
                        print(f"Total: {total}")
                        
                    all_total = all_total + total
                    all_failures = all_failures + failures
                    all_errors = all_errors + errors
                    all_skipped = all_skipped + skipped
                    all_passed = all_passed + passed    
                # Clean up the temporary file
                junit_path.unlink()
        if self.verbose:
            print("All unit tests ran, accumulated results:")
            print(f"Passed: {all_passed}")
            print(f"Failed: {all_failures}")
            print(f"Errors: {all_errors}")
            print(f"Skipped: {all_skipped}")
            print(f"Total: {all_total}")            
        self.queue.put({"total": all_total, "failures": all_failures, "errors": all_errors, "skipped": all_skipped, "passed": all_passed})

def filter_patches_with_p2p_tests(repo, patches, p2p_tests, ref_test_results, logger):
    """
    Validate using unit tests before and after patching. Return True if the patch is valid.
    """
    for i, patch in enumerate(patches):
        # Reset the repo to the original state
        repo.head.reset(index=True, working_tree=True)
        repo.index.checkout(force=True)
        
        try:
            logger.info(f"Testing unit for patch {i}")
            tester = UnitTestRunner(repo_path=REPO_CLONE_PATH, file_paths=p2p_tests, patch=patch, timeout=UNIT_TEST_TIMEOUT, verbose=0, logger=logger)
            tester.start()
            result = tester.join()
        except Exception as e:
            print(f"Error in UnitTestRunner: {e}")
            continue
        finally:
            # Reset the repo to the original state
            repo.head.reset(index=True, working_tree=True)
            repo.index.checkout(force=True)

        if result["total"] == 0:
            continue
            
        if result["failures"] > ref_test_results["failures"] or result["errors"] > ref_test_results["errors"] or result["passed"] < ref_test_results["passed"]:
            continue
        return patch
    return None

# Prediction functions

In [ ]:
def init_logger(instance_i):
    # Create subfolders
    os.makedirs("result", exist_ok=True)
    os.makedirs("result/installer/", exist_ok=True)
    os.makedirs("result/test/", exist_ok=True)
    os.makedirs("result/unit/", exist_ok=True)
    os.makedirs("result/p2p/", exist_ok=True)
        
    installer_logger = setup_logger(f"result/installer/{instance_i - 1}.log")
    test_logger = setup_logger(f"result/test/{instance_i - 1}.log")
    unit_logger = setup_logger(f"result/unit/{instance_i - 1}.log")
    return installer_logger, test_logger, unit_logger


def localize_for_tests(llm, structure, p2p_tests):
    test_related_dir = os.path.join(RESULT_PATH, "related_tests")
    os.makedirs(test_related_dir, exist_ok=True)
    os.makedirs(os.path.join(test_related_dir, "localization_logs"), exist_ok=True)
    rel_args = argparse.Namespace(
        related_level=True,
        output_folder=test_related_dir,
        top_n=3,
        compress_assign=True,
        compress=True,
        start_file=None,
        num_threads=1,
        skip_existing=True,
        backend="local",
        model=llm_model_pth,
        is_test=True,
        # defaults
        output_file=os.path.join(test_related_dir, "loc_outputs.jsonl"),
        file_level=False,
        fine_grain_line_level=False,
        make_summary=False,
        temperature=0.0,
        min_p=MIN_P,
        top_p=TOP_P,
        num_samples=1,
        compress_assign_total_lines=30,
        compress_assign_prefix_lines=10,
        compress_assign_suffix_lines=10,
        merge=False,
        add_space=False,
        no_line_number=False,
        sticky_scroll=False,
        related_level_separate_file=False,
        context_window=10,
        keep_old_order=False,
        irrelevant=False,
        direct_edit_loc=False,
        target_id=None,
        mock=False,
        dataset="princeton-nlp/SWE-bench_Lite",
    )
    return localize(rel_args, llm=llm, structure=structure, start_file_locs=[p2p_tests])[0]

def check_sample_time_limit(sample_start_time):
    if time.time() - sample_start_time > MAX_ALLOWED_TIME_PER_SAMPLE:
        print("Sample time limit exceeded, skipping...")
        return True
    return False

In [ ]:
class TestCreatorRunner:
    def __init__(self, llm_model, nb_samples, structure, related_content, result_dir_name, skip_greedy, timeout=3 * 60):
        self.llm_model = llm_model
        self.nb_samples = nb_samples
        self.structure = structure
        self.related_content = related_content
        self.result_dir_name = result_dir_name
        self.skip_greedy = skip_greedy
        
        self.thread = threading.Thread(target=self.run)
        self.start_time = None
        self.timeout = timeout
        
    def start(self):
        self.thread.start()
        self.start_time = time.time()

    def join(self):
        elapsed_time = time.time() - self.start_time
        self.thread.join(max(0, self.timeout - elapsed_time))
        if self.thread.is_alive():
            return None

    def run(self):
        args = argparse.Namespace(model_path=llm_model_pth, output_base=RESULT_PATH, temperature=TEMPERATURE, min_p=MIN_P, top_p=TOP_P)
        main.generate_tests(args, self.llm_model, self.structure, self.related_content, test_nb_samples=self.nb_samples, result_dir_name=self.result_dir_name, skip_greedy=self.skip_greedy)

In [ ]:
skip_prediction = False
instance_i = 0

def predict_wrapped(problem_statement: str, repo_archive: io.BytesIO, pip_packages_archive: io.BytesIO, env_setup_cmds_templates: list[str]) -> str:
    """ Replace this function with your inference code.
    Args:
        problem_statement: The text of the git issue.
        repo_archive: A BytesIO buffer path with a .tar containing the codebase that must be patched. The gateway will make this directory available immediately before this function runs.
    """
    print("predict: ", problem_statement[:100])
    sample_start_time = time.time()
    global instance_i
    instance_i += 1

    if instance_i == 1:
        while not is_model_ready(llm):
            print("Waiting for model to be ready...")
            time.sleep(10)
        while not is_model_ready(llm_qwq1):
            print("Waiting for model to be ready...")
            time.sleep(10)
        while not is_model_ready(llm_qwq2):
            print("Waiting for model to be ready...")
            time.sleep(10)            
            
    # if "add ModelAdmin.get_inlines() hook to allow" not in problem_statement:
    # if "QTable cannot take" not in problem_statement:
    # if "Migration optimizer does not reduce multiple AlterField" not in problem_statement:
    # if "BUG:" not in problem_statement:
        # return None
    
    if skip_prediction:
        return None

    if time.time() - NOTEBOOK_START_TIME > NOTEBOOK_MAX_ALLOWED_TIME:
        print("Global time limit exceeded, skipping...")
        return None
        
    clean_dirs()

    installer_logger, test_logger, unit_logger = init_logger(instance_i)

    with open('repo_archive.tar', 'wb') as f:
        f.write(repo_archive.read())
    if os.path.exists(REPO_PATH):
        shutil.rmtree(REPO_PATH, ignore_errors=True)
    shutil.unpack_archive('repo_archive.tar', extract_dir=REPO_PATH)
    os.remove('repo_archive.tar')

    ########## Clone the repo
    repo = init_working_repo()

    ######### Create the dummy dataset entry (required by `process_sample` and `generate_tests`)
    data = pd.DataFrame({"instance_id": [instance_i - 1], "repo": [REPO_CLONE_PATH], "problem_statement": [problem_statement], "base_commit": ["dummy"]})
    data.to_parquet(PARQUET_PATH)   
    
    ######### Install the packages
    installer = Installer(pip_packages_archive, env_setup_cmds_templates, timeout=INSTALL_TIMEOUT, repo_path=REPO_CLONE_PATH, logger=installer_logger)
    installer.start()
    print("Started installing packages")
        
    # Get reference unit tests
    structure = get_repo_structure([instance_i - 1], REPO_CLONE_PATH, "dummy_commit", "structure_playground")

    # Start QwQ before related_content
    qwq1_thread = TestCreatorRunner(llm_qwq1, 2, structure, related_content=None, result_dir_name="reproduction_test_samples_qwq1", skip_greedy=True)
    qwq1_thread.start()
    qwq2_thread = TestCreatorRunner(llm_qwq2, 2, structure, related_content=None, result_dir_name="reproduction_test_samples_qwq2", skip_greedy=True)
    qwq2_thread.start()
    
    p2p_tests = get_test_files(instance_id=instance_i - 1, llm=llm, problem_statement=problem_statement, structure=structure, logger_file=f"result/p2p/{instance_i - 1}.log")
    if not p2p_tests:
        print("No pass to pass tests found, skipping...")
        return None
    
    # Get related files for tests
    related_content = localize_for_tests(llm, structure, p2p_tests)
    if not related_content or not related_content.get("found_related_locs", []):
        print("No related content found for tests")
        related_content = None

    ######### Create F2P tests
    # args = argparse.Namespace(model_path=llm_model_pth, output_base=RESULT_PATH, temperature=TEMPERATURE, min_p=MIN_P, top_p=TOP_P)
    # main.generate_tests(args, llm, structure, related_content, test_nb_samples=TEST_SAMPLE_BATCH_SIZE)
    coder_thread = TestCreatorRunner(llm, TEST_NB_SAMPLES, structure, related_content=related_content, result_dir_name="reproduction_test_samples", skip_greedy=False)
    coder_thread.start()

    installer.join()
    qwq1_thread.join()
    qwq2_thread.join()
    coder_thread.join()

    regression_tests = get_regression_tests_that_reproduced_issue(logger=test_logger)
    if not regression_tests:
        print("No valid regression tests found, skipping...")
        return None
         
    # if not regression_tests:
    #     print("No valid regression tests found, retrying...")
    #     nb_retries = (TEST_NB_SAMPLES // TEST_SAMPLE_BATCH_SIZE) - 1
    #     should_return_none = True
    #     for retry_i in range(nb_retries):
    #         retry_temp = TEMPERATURE + (1 + retry_i) * 0.1
    #         print(f"Generating tests for the {retry_i + 1} try with temperature {retry_temp}")
    #         args = argparse.Namespace(model_path=llm_model_pth, output_base=RESULT_PATH, temperature=retry_temp, min_p=MIN_P, top_p=TOP_P)
    #         main.generate_tests(args, llm, structure, related_content, test_nb_samples=TEST_SAMPLE_BATCH_SIZE + 1, skip_greedy=True)
    #         regression_tests = get_regression_tests_that_reproduced_issue(logger=test_logger)
    #         if regression_tests:
    #             should_return_none = False
    #             break
    #     if should_return_none:
    #         print("No valid regression tests found, skipping...")
    #         return None

    if check_sample_time_limit(sample_start_time):
        return None
        
    ######### Get pass to pass tests and run
    unit_logger.info("Getting reference unit tests")
    tester = UnitTestRunner(repo_path=REPO_CLONE_PATH, file_paths=[x.replace("repo_clone/", "") for x in p2p_tests["found_files"]], timeout=UNIT_TEST_TIMEOUT, verbose=0, logger=unit_logger)
    tester.start()
    
    ######### Process the sample to generate repair patches
    args = argparse.Namespace(model_path=llm_model_pth, output_base=RESULT_PATH, temperature=TEMPERATURE, min_p=MIN_P, top_p=TOP_P)
    main.process_sample(args, llm, embd_model, fine_grain_nb_samples=FINE_GRAIN_NB_SAMPLES, repair_nb_samples=REPAIR_NB_SAMPLES, structure=structure, use_memory=USE_MEMORY)
    patches = get_valid_patches()
    ref_test_results = tester.join()

    if not patches:
        print("No valid patches found")
        return None
    print(f"Found {len(patches)} valid patches, testing now...")

    ######### Select the patch with the most regression tests fixed
    patches = select_patch_with_most_regression_tests_fixed(repo, patches, regression_tests, logger=test_logger)
    if not patches:
        print("Patch rejected in F2P tests, skipping...")
        return None
    
    if ref_test_results["passed"] > 0:
        patch = filter_patches_with_p2p_tests(repo, patches, [x.replace("repo_clone/", "") for x in p2p_tests["found_files"]], ref_test_results, logger=unit_logger)
        if not patch:
            print("Patch rejected in P2P tests, skipping...")
            return None
    else:
        patch = patches[0]

    print("Finished processing everything, returning patches...")
    print(patch)
    return patch

In [ ]:
def predict(problem_statement: str, repo_archive: io.BytesIO, pip_packages_archive: io.BytesIO, env_setup_cmds_templates: list[str]) -> str:
    global skip_prediction
    patch_string = None
    start_time = time.time()
    try:
        patch_string = predict_wrapped(problem_statement, repo_archive, pip_packages_archive, env_setup_cmds_templates)
    except Exception as err:
        print(f"Error in predict_wrapped: {err}")
    finally:
        # Always attempt to clean up, even if prediction fails
        print(f"Ran sample in {round((time.time() - start_time) / 60, 2)} minutes")
        clean_dirs()
        if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            skip_prediction = True
    return patch_string

When your notebook is run on the hidden test set, inference_server.serve must be called within 15 minutes of the notebook starting or the gateway will throw an error. If you need more than 15 minutes to load your model you can do so during the very first predict call, which does not have the usual 30 minute response deadline.

In [ ]:
skip_prediction = False

inference_server = kaggle_evaluation.konwinski_prize_inference_server.KPrizeInferenceServer(
    get_number_of_instances,   
    predict
)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        data_paths=(
            '/kaggle/input/konwinski-prize/',  # Path to the entire competition dataset
            '/kaggle/tmp/konwinski-prize/',   # Path to a scratch directory for unpacking data.a_zip.
        ),
        use_concurrency=True,  # This can safely be disabled for purposes of local testing if necessary.
    )

In [ ]:
# !zip result_kaggle.zip -r result

In [ ]:
stop_llama_server(server_process0)
stop_llama_server(server_process1)
stop_llama_server(server_process2)

In [ ]:
# Clean everything except submission.csv
files = os.listdir("/kaggle/working/")
for path_str in files:
    if path_str.endswith("submission.csv"):
        continue
    path = Path("/kaggle/working/") / path_str
    try:
        if path.exists():
            if path.is_file():
                path.unlink()
            else:
                shutil.rmtree(path, ignore_errors=True)
    except Exception as e:
        print(f"Error while removing {path}: {e}")
        continue